# Experimento 2 — YOLOv8n @ imgsz=960 (INDIGO)

Segundo experimento: **misma arquitectura YOLOv8n**, único cambio principal **`imgsz=960`**
frente al baseline a 640 px.

**Hipótesis:** mayor resolución de entrada mejora la detección de grietas (`crack`) pequeñas.

**Requisitos previos**

- Mismo ZIP en Drive: `/content/drive/MyDrive/egg-detection/datasets/indigo_yolo.zip`
- Baseline completado en `runs/yolov8n_baseline/` (no sobrescribir).
- Runtime **GPU** (T4 o superior).

**Run de este experimento:** `/content/drive/MyDrive/egg-detection/runs/yolov8n_960/`

Documentación: [`docs/18-experimento-yolov8n-960.md`](../docs/18-experimento-yolov8n-960.md)

## 1. Verificar entorno

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Plataforma:", platform.platform())

try:
    import torch

    cuda_ok = torch.cuda.is_available()
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", cuda_ok)
    if cuda_ok:
        print("GPU:", torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f"Memoria GPU total: {props.total_memory / (1024**3):.2f} GiB")
    else:
        print("\n⚠️  ADVERTENCIA: No hay GPU disponible.")
        print("   Activa Runtime → Change runtime type → GPU antes de entrenar.")
except ImportError:
    print("PyTorch aún no instalado (se instalará con Ultralytics).")

print("\n--- nvidia-smi ---")
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi no disponible (¿runtime sin GPU?)")

## 2. Instalar dependencias

In [ ]:
!pip install -q ultralytics

import ultralytics

print("Ultralytics:", ultralytics.__version__)

## 3. Montar Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DATASET_ZIP = Path("/content/drive/MyDrive/egg-detection/datasets/indigo_yolo.zip")
DATASET_ROOT = Path("/content/indigo_yolo")
RUNS_ROOT = Path("/content/drive/MyDrive/egg-detection/runs")
RUN_NAME = "yolov8n_960"

print("DATASET_ZIP:", DATASET_ZIP, "→ existe:", DATASET_ZIP.exists())
print("DATASET_ROOT:", DATASET_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)
print("RUN_NAME:", RUN_NAME)

## 4. Extraer dataset a `/content` y verificar

Conteos esperados: **629 train**, **111 val**, **100 test**.

In [ ]:
import shutil
import zipfile

import yaml

EXPECTED = {"train": 629, "val": 111, "test": 100}
REQUIRED_DIRS = [
    "images/train", "images/val", "images/test",
    "labels/train", "labels/val", "labels/test",
]


def contar_dataset(root: Path) -> dict[str, dict[str, int]]:
    counts = {}
    for split in EXPECTED:
        n_img = len(list((root / "images" / split).glob("*.jpg")))
        n_lbl = len(list((root / "labels" / split).glob("*.txt")))
        counts[split] = {"images": n_img, "labels": n_lbl}
    return counts


def validar_conteos(counts: dict[str, dict[str, int]]) -> list[str]:
    errors = []
    for split, exp in EXPECTED.items():
        if counts[split]["images"] != exp:
            errors.append(f"{split}: imgs={counts[split]['images']} (esperado {exp})")
        if counts[split]["labels"] != exp:
            errors.append(f"{split}: labels={counts[split]['labels']} (esperado {exp})")
    return errors


def dataset_ya_valido(root: Path) -> bool:
    if not root.exists():
        return False
    if any(not (root / d).exists() for d in REQUIRED_DIRS):
        return False
    return len(validar_conteos(contar_dataset(root))) == 0


def encontrar_raiz_yolo(base: Path) -> Path:
    """Busca recursivamente una carpeta con images/train y labels/train."""
    if (base / "images" / "train").is_dir() and (base / "labels" / "train").is_dir():
        return base
    for images_dir in sorted(base.rglob("images")):
        if not images_dir.is_dir():
            continue
        root = images_dir.parent
        if (images_dir / "train").is_dir() and (root / "labels" / "train").is_dir():
            return root
    raise FileNotFoundError(
        "Estructura YOLO no encontrada tras extraer ZIP "
        "(se esperaba images/train y labels/train)."
    )


def extraer_desde_zip(zip_path: Path, destino: Path) -> None:
    """Extrae ZIP normalizando rutas Windows (\\ -> /) creadas en Windows."""
    tmp = Path("/content/_indigo_extract_tmp")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)

    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            member = info.filename.replace("\\", "/")
            if member.endswith("/"):
                continue
            out_path = tmp / member
            out_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(out_path, "wb") as dst:
                shutil.copyfileobj(src, dst)

    raiz = encontrar_raiz_yolo(tmp)
    if destino.exists():
        shutil.rmtree(destino)
    shutil.move(str(raiz), str(destino))
    shutil.rmtree(tmp, ignore_errors=True)


if dataset_ya_valido(DATASET_ROOT):
    print("✅ Dataset ya válido en /content — omitiendo extracción.")
else:
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f"ZIP no encontrado: {DATASET_ZIP}")
    zip_bytes = DATASET_ZIP.stat().st_size
    _, _, free = shutil.disk_usage("/content")
    espacio_req = max(int(zip_bytes * 1.2), int(2.6 * 1024**3))
    print(f"ZIP: {zip_bytes / (1024**3):.3f} GiB | libre /content: {free / (1024**3):.3f} GiB")
    if free < espacio_req:
        raise RuntimeError("Espacio insuficiente en /content.")
    extraer_desde_zip(DATASET_ZIP, DATASET_ROOT)

counts = contar_dataset(DATASET_ROOT)
for split, c in counts.items():
    print(f"  {split}: {c['images']} imgs, {c['labels']} labels")
errors = validar_conteos(counts)
if errors:
    raise ValueError("Conteos inválidos:\n" + "\n".join(errors))
print("✅ Dataset verificado.")

DATA_YAML = DATASET_ROOT / "data_colab.yaml"
yaml_payload = {
    "path": "/content/indigo_yolo",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 2,
    "names": {0: "egg", 1: "crack"},
}
DATA_YAML.write_text(yaml.dump(yaml_payload, default_flow_style=False), encoding="utf-8")
print("data_colab.yaml:", DATA_YAML)

## 5. Configuración del experimento

Variables controladas vs baseline. **Único cambio principal:** `IMGSZ = 960`.

In [ ]:
import os
import random
import time

import numpy as np
import torch

SEED = 42
IMGSZ = 960
EPOCHS = 50
PATIENCE = 10
WORKERS = 2

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Experimento: YOLOv8n | imgsz={IMGSZ} | epochs={EPOCHS} | seed={SEED}")

## 6. Modelo YOLOv8n

Misma arquitectura que el baseline. **No** usar YOLOv8s.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
print("Modelo:", "yolov8n")

## 7. Entrenamiento

`batch=-1` (AutoBatch). Tras entrenar se intenta leer el batch efectivo desde `args.yaml`.
Si `args.yaml` conserva `-1`, **no** se registra como batch real (queda `null`).

In [ ]:
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("Activa GPU en Colab (Runtime → Change runtime type).")

train_start = time.time()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,
    patience=PATIENCE,
    seed=SEED,
    device=0,
    workers=WORKERS,
    project=str(RUNS_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

train_elapsed_s = time.time() - train_start
RUN_DIR = RUNS_ROOT / RUN_NAME
print("RUN_DIR:", RUN_DIR)
print(f"Tiempo entrenamiento (wall clock): {train_elapsed_s / 3600:.3f} h")

args_yaml = RUN_DIR / "args.yaml"
batch_from_args = None
if args_yaml.exists():
    args_saved = yaml.safe_load(args_yaml.read_text(encoding="utf-8"))
    batch_from_args = args_saved.get("batch")

# -1 significa "auto" en Ultralytics; no es el batch efectivo en GPU
if isinstance(batch_from_args, int) and batch_from_args > 0:
    batch_real = batch_from_args
    batch_note = None
    print(f"Batch efectivo (args.yaml): {batch_real}")
else:
    batch_real = None
    batch_note = (
        "AutoBatch solicitado (batch=-1). El batch efectivo no quedó en args.yaml; "
        "revisar log de entrenamiento ('AutoBatch') si se necesita el valor exacto."
    )
    print(f"Batch en args.yaml: {batch_from_args!r} → efectivo no capturado")
    print(batch_note)

## 8. Checkpoints

In [ ]:
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

for path in (BEST_PT, LAST_PT):
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint no encontrado: {path}")
    print(f"✅ {path.name}: {path.stat().st_size / (1024**2):.1f} MiB")

model = YOLO(str(BEST_PT))

## 9. Métricas de validación (global + por clase)

API Ultralytics: `box.mp/mr/map50/map` (global), `class_result(i)` y `summary()` (por clase).

In [ ]:
val_metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    device=0,
    plots=True,
    save_json=True,
    verbose=True,
)


def extract_box_metrics(metrics):
    box = metrics.box
    return {
        "precision": float(box.mp),
        "recall": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }


def extract_per_class(metrics):
    """Métricas por clase desde DetMetrics (Ultralytics)."""
    out = {}
    for i, cls_idx in enumerate(metrics.box.ap_class_index):
        name = metrics.names[int(cls_idx)]
        p, r, ap50, ap = metrics.box.class_result(i)
        out[name] = {
            "precision": float(p),
            "recall": float(r),
            "mAP50": float(ap50),
            "mAP50_95": float(ap),
        }
    return out


val_scores = extract_box_metrics(val_metrics)
val_per_class = extract_per_class(val_metrics)
val_speed = dict(val_metrics.speed) if hasattr(val_metrics, "speed") else {}

print("=== VALIDATION global ===")
for k, v in val_scores.items():
    print(f"  {k}: {v:.4f}")
print("\n=== VALIDATION por clase ===")
for cls_name, m in val_per_class.items():
    print(f"  {cls_name}: P={m['precision']:.4f} R={m['recall']:.4f} "
          f"mAP50={m['mAP50']:.4f} mAP50-95={m['mAP50_95']:.4f}")

## 10. Test final (una sola vez)

⚠️ Split **test** reservado — no usar para ajustar hiperparámetros.

In [ ]:
test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMGSZ,
    device=0,
    plots=True,
    save_json=True,
    verbose=True,
)

test_scores = extract_box_metrics(test_metrics)
test_per_class = extract_per_class(test_metrics)
test_speed = dict(test_metrics.speed) if hasattr(test_metrics, "speed") else {}

print("=== TEST global (evaluación única) ===")
for k, v in test_scores.items():
    print(f"  {k}: {v:.4f}")
print("\n=== TEST por clase ===")
for cls_name, m in test_per_class.items():
    print(f"  {cls_name}: P={m['precision']:.4f} R={m['recall']:.4f} "
          f"mAP50={m['mAP50']:.4f} mAP50-95={m['mAP50_95']:.4f}")

## 11. Matrices de confusión y curvas

In [ ]:
from IPython.display import Image as IPyImage, display

plot_names = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png", "PR_curve.png",
    "BoxF1_curve.png", "F1_curve.png",
    "BoxP_curve.png", "P_curve.png",
    "BoxR_curve.png", "R_curve.png",
]
shown = set()
for name in plot_names:
    for path in [RUN_DIR / name, *RUN_DIR.rglob(name)]:
        if path.exists() and path not in shown:
            shown.add(path)
            print(path.name)
            display(IPyImage(filename=str(path)))

## 12. Curvas de entrenamiento

In [ ]:
results_png = RUN_DIR / "results.png"
if results_png.exists():
    display(IPyImage(filename=str(results_png)))

results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    import pandas as pd

    df_epochs = pd.read_csv(results_csv)
    df_epochs.columns = [c.strip() for c in df_epochs.columns]
    display(df_epochs.tail(10))

## 13. Predicciones visuales (test)

In [ ]:
import matplotlib.pyplot as plt

test_img_dir = DATASET_ROOT / "images" / "test"
test_images = sorted(test_img_dir.glob("*.jpg"))
sample_paths = random.Random(SEED).sample(test_images, min(6, len(test_images)))

predictions = model.predict(
    source=[str(p) for p in sample_paths],
    imgsz=IMGSZ,
    conf=0.25,
    save=False,
    verbose=False,
)

for src_path, result in zip(sample_paths, predictions):
    plt.figure(figsize=(8, 10))
    plt.imshow(result.plot()[:, :, ::-1])
    plt.title(src_path.name)
    plt.axis("off")
    plt.show()

## 14. Coste computacional

Registrar tiempo, batch, velocidad de inferencia y tamaño del checkpoint.

In [ ]:
best_pt_mb = BEST_PT.stat().st_size / (1024**2)

print(f"Tiempo entrenamiento: {train_elapsed_s / 3600:.3f} h ({train_elapsed_s:.0f} s)")
print(f"Batch AutoBatch: {batch_real}")
print(f"Tamaño best.pt: {best_pt_mb:.2f} MiB")
print(f"Velocidad val (ms/img): {val_speed}")
print(f"Velocidad test (ms/img): {test_speed}")

if torch.cuda.is_available():
    print(f"Memoria GPU reservada: {torch.cuda.max_memory_allocated() / (1024**3):.3f} GiB")

## 15. Comparación vs baseline (640 px)

Diferencias absolutas del experimento 960 frente al baseline documentado.
**No se declara automáticamente que 960 sea mejor** — la conclusión depende de los resultados.

In [ ]:
import pandas as pd

BASELINE = {
    "val": {
        "precision": 0.9004044862968609,
        "recall": 0.8160000000000001,
        "mAP50": 0.8617328825985284,
        "mAP50_95": 0.6816909837862021,
    },
    "test": {
        "precision": 0.8875965337677447,
        "recall": 0.8203905380843488,
        "mAP50": 0.844116054312489,
        "mAP50_95": 0.6725147798273655,
    },
}


def comparar(split_name, exp_scores, base_scores):
    rows = []
    for metric in ("precision", "recall", "mAP50", "mAP50_95"):
        b = base_scores[metric]
        e = exp_scores[metric]
        rows.append({
            "split": split_name,
            "metric": metric,
            "baseline_640": b,
            "exp_960": e,
            "delta_abs": e - b,
        })
    return rows


rows = comparar("val", val_scores, BASELINE["val"])
rows += comparar("test", test_scores, BASELINE["test"])

df_cmp = pd.DataFrame(rows)
display(df_cmp)

if "crack" in val_per_class:
    c = val_per_class["crack"]
    print("\n=== Foco crack (val) ===")
    print(f"  recall={c['recall']:.4f} | mAP50={c['mAP50']:.4f} | mAP50-95={c['mAP50_95']:.4f}")
    print("  (Comparar con log baseline: crack R≈0.640, mAP50≈0.731, mAP50-95≈0.369)")

print("\nInterpretación: revisar especialmente delta de recall/mAP de crack y coste computacional.")

## 16. Exportar resumen (JSON + CSV)

In [ ]:
import csv
import json
from datetime import datetime, timezone


def flatten_per_class(prefix, per_class_dict):
    flat = {}
    for cls_name, m in per_class_dict.items():
        for mk, mv in m.items():
            flat[f"{prefix}_{cls_name}_{mk}"] = mv
    return flat


summary = {
    "experimento": "yolov8n_960",
    "modelo": "yolov8n",
    "fecha_utc": datetime.now(timezone.utc).isoformat(),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch_requested": "auto (-1)",
    "batch_real": batch_real,
    "batch_note": batch_note,
    "patience": PATIENCE,
    "seed": SEED,
    "train_elapsed_s": train_elapsed_s,
    "best_pt_mb": best_pt_mb,
    "dataset_root": str(DATASET_ROOT),
    "data_yaml": str(DATA_YAML),
    "precision_val": val_scores["precision"],
    "recall_val": val_scores["recall"],
    "mAP50_val": val_scores["mAP50"],
    "mAP50_95_val": val_scores["mAP50_95"],
    "precision_test": test_scores["precision"],
    "recall_test": test_scores["recall"],
    "mAP50_test": test_scores["mAP50"],
    "mAP50_95_test": test_scores["mAP50_95"],
    "val_speed_ms": val_speed,
    "test_speed_ms": test_speed,
    "best_pt": str(BEST_PT),
    "last_pt": str(LAST_PT),
    "run_dir": str(RUN_DIR),
    "ultralytics_version": ultralytics.__version__,
    "baseline_reference": "runs/yolov8n_baseline @ imgsz=640",
    **flatten_per_class("val", val_per_class),
    **flatten_per_class("test", test_per_class),
}

summary_json = RUN_DIR / "training_summary.json"
summary_csv = RUN_DIR / "training_summary.csv"

summary_json.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

with summary_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(summary.keys()))
    writer.writeheader()
    writer.writerow(summary)

print("Guardado:", summary_json)
print(json.dumps(summary, indent=2, ensure_ascii=False)[:2000], "...")